<a href="https://colab.research.google.com/github/fadhil-code/-2021/blob/main/Enhanced_Fractal_Encryption_Techniques_Modern_Innovations_and_Optimization_Strategies_for_Secure_Image_Protection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install numpy pillow matplotlib scikit-image opencv-python pyfftw --quiet

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim
import cv2
import time
import math
import io
import os

In [ ]:
def to_grayscale(img):
    if img.ndim == 3:
        return cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    return img

def load_image(path, target_size=None, as_gray=True):
    img = Image.open(path).convert('RGB')
    if target_size:
        img = img.resize(target_size, Image.BICUBIC)
    arr = np.array(img)
    return to_grayscale(arr) if as_gray else arr

def show_images(images, titles=None, cmap='gray', figsize=(12,6)):
    n = len(images)
    plt.figure(figsize=figsize)
    for i,img in enumerate(images):
        plt.subplot(1,n,i+1)
        if img.ndim==2:
            plt.imshow(np.clip(img,0,255).astype(np.uint8), cmap=cmap)
        else:
            plt.imshow(np.clip(img,0,255).astype(np.uint8))
        if titles: plt.title(titles[i])
        plt.axis('off')
    plt.show()

def fractal_mask(shape, center=(-0.5,0.0), scale=1.5, max_iter=100, power=2, seed_shift=0.0):
    h,w = shape
    x = np.linspace(center[0]-scale, center[0]+scale, w)
    y = np.linspace(center[1]-scale*(h/w), center[1]+scale*(h/w), h)
    X,Y = np.meshgrid(x,y)
    C = X + 1j*Y + (seed_shift + 0j)
    Z = np.zeros_like(C)
    M = np.zeros(C.shape, dtype=np.float32)
    for i in range(max_iter):
        Z = Z**power + C
        escaped = np.abs(Z) > 2.0
        newly = escaped & (M==0)
        M[newly] = i
        Z[escaped] = 0
    M[M==0] = max_iter
    M = (M - M.min()) / (M.max() - M.min() + 1e-12)
    return M

In [ ]:
def make_key(center_x, center_y, scale, max_iter, power, seed_shift, phase_shift):
    return {
        'center': (float(center_x), float(center_y)),
        'scale': float(scale),
        'max_iter': int(max_iter),
        'power': float(power),
        'seed_shift': float(seed_shift),
        'phase_shift': float(phase_shift)
    }

def encrypt_fractal_fft(img_gray, key, complex_mode=True):
    img = img_gray.astype(np.float32)
    h,w = img.shape
    F = np.fft.fft2(img)
    Fshift = np.fft.fftshift(F)
    mask_real = fractal_mask((h,w), center=key['center'], scale=key['scale'],
                             max_iter=key['max_iter'], power=key['power'],
                             seed_shift=key['seed_shift'])
    if complex_mode:
        phase = (mask_real * 2 * np.pi + key['phase_shift']) % (2*np.pi)
        complex_mask = np.exp(1j * phase) * (1 + 0.5*mask_real)
        F_mod = Fshift * complex_mask
    else:
        real_mask = 1.0 + 0.9*mask_real
        F_mod = Fshift * real_mask
    F_ishift = np.fft.ifftshift(F_mod)
    img_enc = np.fft.ifft2(F_ishift)
    img_enc_real = np.real(img_enc)
    enc_norm = img_enc_real - img_enc_real.min()
    enc_norm = enc_norm / (enc_norm.max()+1e-12) * 255.0
    return enc_norm.astype(np.uint8)

def decrypt_fractal_fft(enc_img_uint8, key, complex_mode=True):
    enc = enc_img_uint8.astype(np.float32)
    F = np.fft.fft2(enc)
    Fshift = np.fft.fftshift(F)
    h,w = enc.shape
    mask_real = fractal_mask((h,w), center=key['center'], scale=key['scale'],
                             max_iter=key['max_iter'], power=key['power'],
                             seed_shift=key['seed_shift'])
    if complex_mode:
        phase = (mask_real * 2 * np.pi + key['phase_shift']) % (2*np.pi)
        complex_mask = np.exp(1j * phase) * (1 + 0.5*mask_real)
        complex_mask[np.abs(complex_mask) < 1e-6] = 1e-6
        F_rec = Fshift / complex_mask
    else:
        real_mask = 1.0 + 0.9*mask_real
        real_mask[real_mask==0] = 1e-6
        F_rec = Fshift / real_mask
    F_ishift = np.fft.ifftshift(F_rec)
    img_rec = np.fft.ifft2(F_ishift)
    img_rec_real = np.real(img_rec)
    rec = img_rec_real - img_rec_real.min()
    rec = rec / (rec.max()+1e-12) * 255.0
    return np.clip(rec,0,255).astype(np.uint8)

def encrypt_xor(img_gray, key_bytes=0xAA):
    arr = img_gray.astype(np.uint8)
    return (arr ^ (key_bytes & 0xFF)).astype(np.uint8)

def decrypt_xor(enc, key_bytes=0xAA):
    return (enc ^ (key_bytes & 0xFF)).astype(np.uint8)

def encrypt_permute(img_gray, permutation_seed=12345):
    h,w = img_gray.shape
    flat = img_gray.flatten()
    rng = np.random.RandomState(permutation_seed)
    perm = rng.permutation(len(flat))
    enc = flat[perm]
    return enc.reshape(h,w), perm

In [ ]:
def decrypt_permute(enc_img, perm):
    h,w = enc_img.shape
    flat = enc_img.flatten()
    inv = np.empty_like(flat)
    inv[perm] = flat
    return inv.reshape(h,w)

def compute_metrics(original, recovered):
    orig = original.astype(np.float32)
    rec = recovered.astype(np.float32)
    mse = np.mean((orig - rec)**2)
    p = psnr(orig, rec, data_range=255)
    s = ssim(orig, rec, data_range=255)
    return {'MSE': float(mse), 'PSNR': float(p), 'SSIM': float(s)}

try:
    sample_path = 'sample_input.png'
    if not os.path.exists(sample_path):
        w,h = 512,512
        grad = np.tile(np.linspace(0,255,w,dtype=np.uint8),(h,1))
        circles = np.zeros((h,w), dtype=np.uint8)
        cv2.circle(circles, (256,256), 100, 180, -1)
        text = np.zeros((h,w), dtype=np.uint8)
        cv2.putText(text, 'FENIX', (120,300), cv2.FONT_HERSHEY_SIMPLEX, 3, 255, 6)
        sample = np.clip(0.6*grad + 0.4*circles + 0.6*text,0,255).astype(np.uint8)
        Image.fromarray(sample).save(sample_path)
    img = load_image(sample_path, as_gray=True)
except Exception as e:
    raise RuntimeError("Provide an input image named 'sample_input.png' in the runtime or modify the script.") from e

show_images([img], ['Original (grayscale)'])

key = make_key(center_x=-0.6, center_y=0.0, scale=1.2, max_iter=120, power=2.0, seed_shift=0.01, phase_shift=0.5)

t0 = time.time()
enc_fractal = encrypt_fractal_fft(img, key, complex_mode=True)
t1 = time.time()
dec_fractal = decrypt_fractal_fft(enc_fractal, key, complex_mode=True)
t2 = time.time()
metrics_fractal = compute_metrics(img, dec_fractal)
time_enc_fractal = t1 - t0
time_dec_fractal = t2 - t1

t0 = time.time()
enc_xor = encrypt_xor(img, key_bytes=0xAB)
t1 = time.time()
dec_xor = decrypt_xor(enc_xor, key_bytes=0xAB)
t2 = time.time()
metrics_xor = compute_metrics(img, dec_xor)
time_enc_xor = t1 - t0
time_dec_xor = t2 - t1

t0 = time.time()
enc_perm, perm = encrypt_permute(img, permutation_seed=2025)
t1 = time.time()
dec_perm = decrypt_permute(enc_perm, perm)
t2 = time.time()
metrics_perm = compute_metrics(img, dec_perm)
time_enc_perm = t1 - t0
time_dec_perm = t2 - t1

show_images([enc_fractal, enc_xor, enc_perm], ['Fractal-FFT Encrypted', 'XOR Encrypted', 'Permuted Encrypted'])
show_images([dec_fractal, dec_xor, dec_perm], ['Fractal-FFT Decrypted', 'XOR Decrypted', 'Permutation Decrypted'])

print("=== Metrics ===")
print("Fractal-FFT:", metrics_fractal, f"enc_time={time_enc_fractal:.4f}s dec_time={time_dec_fractal:.4f}s")
print("XOR:", metrics_xor, f"enc_time={time_enc_xor:.6f}s dec_time={time_dec_xor:.6f}s")
print("Permutation:", metrics_perm, f"enc_time={time_enc_perm:.6f}s dec_time={time_dec_perm:.6f}s")

def key_perturb(original_key, epsilon=1e-3, field='seed_shift'):
    k = original_key.copy()
    k[field] = k[field] + epsilon
    return k

In [ ]:
perturbed_key = key_perturb(key, epsilon=0.001, field='seed_shift')
dec_bad = decrypt_fractal_fft(enc_fractal, perturbed_key, complex_mode=True)
metrics_perturbed = compute_metrics(img, dec_bad)
show_images([dec_fractal, dec_bad], ['Decrypted (correct key)', 'Decrypted (perturbed key)'])
print("Metrics with perturbed key:", metrics_perturbed)

Image.fromarray(enc_fractal).save('enc_fractal.png')
Image.fromarray(dec_fractal).save('dec_fractal.png')